# Phase 1.2 : Preprocessing — normalisation + batching

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase1_1_setup.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait rapidement le setup de la phase 1.1 (télécharge et réorganise le dataset), sans les explications déjà vues. Si tu es toujours dans le même runtime Colab que la phase 1.1 (mêmes variables en mémoire), tu peux sauter directement à la section "Phase 1.2".

## Reprise (setup phase 1.1, condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [2]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [3]:
!pip install kaggle -q
!mkdir -p ~/.kaggle


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
La syntaxe de la commande n'est pas correcte.


In [4]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
from PIL import Image

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


def list_valid_images(folder):
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            with Image.open(path) as img:
                img.verify()
            valid.append(path)
        except Exception:
            continue
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

## Phase 1.2 : Preprocessing : normalisation + batching

**Objectif** : charger les images avec `image_dataset_from_directory`, normaliser les valeurs de pixels, configurer le batching, et afficher le premier batch pour vérifier les shapes.

In [ ]:
import tensorflow as tf

# IMG_SIZE : 128x128 est un bon compromis qualité/RAM pour ce dataset sur Colab T4.
# BATCH_SIZE : 32 est le défaut standard.
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)

### Normalisation + cache/prefetch

On ramène les pixels dans `[0, 1]` avec `Rescaling(1./255)`, puis on optimise le pipeline avec `.cache()` (évite de relire le disque à chaque epoch) et `.prefetch()` (chevauche préparation des données et calcul GPU).

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Vérification : shape et valeurs du premier batch
images, labels = next(iter(train_ds))
print(f"Shape images batch : {images.shape}")
print(f"Shape labels batch : {labels.shape}")
print(f"Valeurs min / max : {images.numpy().min():.2f} / {images.numpy().max():.2f}")

### Checklist avant la phase 1.3

- **Happy path** : shape `(32, 128, 128, 3)` pour les images, `(32, 1)` pour les labels, valeurs entre 0.0 et 1.0.
- **Edge case** : oublier `.cache()` sur `train_ds` → le pipeline relit le disque à chaque epoch, l'entraînement devient 3-5x plus lent (visible sur le temps par epoch).
- **Adversarial** : un dossier parasite (ex. `.ipynb_checkpoints`) dans `train/` ferait planter `image_dataset_from_directory` avec plus de 2 classes détectées. On a déjà vérifié la structure en phase 1.1, donc ce n'est pas un risque ici.